In [1]:
import pandas as pd
import numpy as np
from math import floor

# ---------------------------------------------------------
# Premissas:
# - df: DataFrame com colunas ['music_id', 'music_frase_id', 'frase', 'frase_id', 'tem_padrao', 'padrao_encontrado']
# - tem_padrao ∈ {'N', 'Female', 'Male'}
# ---------------------------------------------------------

PATH = "data/frases-musicas-final-filled.csv"
OUTPUT_PATH = "data/assignments.csv"

# Reprodutibilidade
RNG_SEED = 42
rng = np.random.default_rng(RNG_SEED)

# Parâmetros da distribuição
N_ANNOTATORS = 35
ASSIGN_PER_ANNOTATOR = 2000
N_PER_ANNOTATOR = 100               # sem padrão por anotador
PATTERN_PER_ANNOTATOR = ASSIGN_PER_ANNOTATOR - N_PER_ANNOTATOR  # 1900

# Proporções desejadas entre Female/Male dentro dos "com padrão"
# (use as suas proporções globais; aqui estão aproximadas a partir do enunciado)
FEMALE_PROP = 0.004213 / (0.004213 + 0.002762)  # ≈ 0.604
MALE_PROP   = 1.0 - FEMALE_PROP                 # ≈ 0.396

# Alvos por anotador (arredonda e corrige para somar PATTERN_PER_ANNOTATOR)
female_target = int(round(PATTERN_PER_ANNOTATOR * FEMALE_PROP))
male_target   = PATTERN_PER_ANNOTATOR - female_target



In [2]:
df = pd.read_csv(PATH)
df

,music_id,music_frase_id,frase,frase_id,tem_padrao,padrao_encontrado
0,1,1,Carolina é uma menina bem difícil de esquecer.,1,Female,"[(1916877163388338700, 3, 6), (191687716338833..."
1,1,2,Andar bonito e um brilho no olhar.,2,N,NaN
2,1,3,Tem um jeito adolescente que me faz enlouquecer.,3,N,NaN
3,1,4,E um molejo que eu não vou te enganar.,4,N,NaN
4,1,5,"Maravilha feminina, meu docinho de pavê.",5,N,NaN
...,...,...,...,...,...,...
2685690,146609,480,Recite as afirmações por 10 minutos ininterrup...,4574304,N,NaN
2685691,146611,4,Chove sem parar\n.,4574434,N,NaN
2685692,146611,10,De molhar o meu divino amor\n.,4574440,N,NaN
2685693,146611,14,Inocente como a flor\n.,4574444,N,NaN


In [3]:
# ---- Separa listas de IDs por classe ----
df = df.copy()
df['tem_padrao'] = df['tem_padrao'].astype(str)

ids_N      = df.loc[df['tem_padrao'] == 'N', 'frase_id'].drop_duplicates().to_numpy()
ids_Female = df.loc[df['tem_padrao'] == 'Female', 'frase_id'].drop_duplicates().to_list()
ids_Male   = df.loc[df['tem_padrao'] == 'Male', 'frase_id'].drop_duplicates().to_list()

# Checagens mínimas
if len(ids_N) < N_PER_ANNOTATOR:
    raise ValueError(f"Faltam sentenças 'N'. Precisa de pelo menos {N_PER_ANNOTATOR}, tem {len(ids_N)}.")

# Para maximizar sobreposição dos "N":
# - escolhemos um pool fixo de 100 sentenças N e damos as MESMAS 100 a todos os annotators.
ids_N_pool = rng.choice(ids_N, size=N_PER_ANNOTATOR, replace=False).tolist()

# Para os com padrão:
# - Precisamos cobrir todos Female e Male ao menos 1x;
# - Temos capacidade total de padrão: N_ANNOTATORS * PATTERN_PER_ANNOTATOR = 66.500
# - Únicas c/ padrão: len(ids_Female) + len(ids_Male) ≈ 18.733
# - Sobra capacidade para múltiplas anotações (sobreposição).

# Embaralha listas para não viciar a ordem
rng.shuffle(ids_Female)
rng.shuffle(ids_Male)

# Vamos ciclar pelas listas quantas vezes forem necessárias (round-robin).
def round_robin_cycle(lst):
    """Itera infinitamente sobre a lista (ciclo)."""
    i = 0
    L = len(lst)
    while True:
        yield lst[i % L]
        i += 1

female_cycle = round_robin_cycle(ids_Female) if ids_Female else None
male_cycle   = round_robin_cycle(ids_Male)   if ids_Male   else None

# Inicializa estruturas
annotators = list(range(1, N_ANNOTATORS + 1))
assignments = {a: [] for a in annotators}

# 1) Atribui as 100 sentenças N iguais para todos (max sobreposição nas "N")
for a in annotators:
    assignments[a].extend(ids_N_pool)

# 2) Atribui padrões respeitando targets por anotador e garantindo cobertura total
#    Estratégia:
#    - Primeiro garante cobertura 1x de todos Female e Male (passada de cobertura).
#    - Depois preenche até bater female_target/male_target de cada anotador, usando round-robin.

# 2a) Cobertura mínima de todos Female (se houver)
if ids_Female:
    # Distribui 1x cada Female em rodízio pelos anotadores
    idx = 0
    for fid in ids_Female:
        a = annotators[idx % N_ANNOTATORS]
        assignments[a].append(fid)
        idx += 1

# 2b) Cobertura mínima de todos Male (se houver)
if ids_Male:
    idx = 0
    for mid in ids_Male:
        a = annotators[idx % N_ANNOTATORS]
        assignments[a].append(mid)
        idx += 1

# 2c) Agora completa cada anotador até os alvos por classe, com round-robin e sobreposição
from collections import Counter

def count_by_class(ids):
    # ajuda a ver Female/Male já atribuídos por anotador
    return Counter(df.loc[df['frase_id'].isin(ids), 'tem_padrao'])

for a in annotators:
    current_ids = set(assignments[a])
    # Conta quantos Female/Male já tem
    cnt = count_by_class(list(current_ids))
    cf = cnt.get('Female', 0)
    cm = cnt.get('Male', 0)

    # Completa Female até female_target
    while cf < female_target and female_cycle is not None:
        fid = next(female_cycle)
        # permite sobreposição: não impedimos repetição entre anotadores,
        # mas evitamos duplicar a MESMA frase no mesmo anotador.
        if fid not in current_ids:
            assignments[a].append(fid)
            current_ids.add(fid)
            cf += 1

    # Completa Male até male_target
    while cm < male_target and male_cycle is not None:
        mid = next(male_cycle)
        if mid not in current_ids:
            assignments[a].append(mid)
            current_ids.add(mid)
            cm += 1

    # Se por algum motivo ainda não chegou a 2000 (ex.: listas vazias),
    # preenche com N extras (permitindo sobreposição interna mínima).
    # Mas mantendo a maior sobreposição possível, usamos o mesmo pool_N.
    while len(assignments[a]) < ASSIGN_PER_ANNOTATOR:
        # recicla N do pool (evita estourar caso extremo)
        assignments[a].append(ids_N_pool[(len(assignments[a]) - N_PER_ANNOTATOR) % len(ids_N_pool)])

# 3) Converte para DataFrame final (annotator_id, frase_id)
rows = []
for a in annotators:
    # garantias finais
    unique_ids = []
    seen = set()
    for fid in assignments[a]:
        if fid not in seen:
            seen.add(fid)
            unique_ids.append(fid)
        if len(unique_ids) == ASSIGN_PER_ANNOTATOR:
            break
    # Se por acaso retiramos duplicatas internas e ficamos com menos, completa com N_pool
    i = 0
    while len(unique_ids) < ASSIGN_PER_ANNOTATOR:
        candidate = ids_N_pool[i % len(ids_N_pool)]
        if candidate not in seen:
            unique_ids.append(candidate)
            seen.add(candidate)
        i += 1

    rows.extend([(a, fid) for fid in unique_ids])

assignments_df = pd.DataFrame(rows, columns=['annotator_id', 'frase_id'])

# --------- Checagens úteis (opcionais) ----------
# 1) Tamanho por anotador
check_counts = assignments_df.groupby('annotator_id').size()
assert (check_counts == ASSIGN_PER_ANNOTATOR).all()

# 2) Cada anotador tem 100 'N'?
merged = assignments_df.merge(df[['frase_id', 'tem_padrao']], on='frase_id', how='left')
n_per_ann = merged[merged['tem_padrao'] == 'N'].groupby('annotator_id').size()
assert (n_per_ann == N_PER_ANNOTATOR).all()

# 3) Proporção Female/Male por anotador próxima da meta
pat_per_ann = merged[merged['tem_padrao'].isin(['Female', 'Male'])].groupby(['annotator_id', 'tem_padrao']).size().unstack(fill_value=0)
pat_per_ann['female_ratio'] = pat_per_ann['Female'] / (pat_per_ann['Female'] + pat_per_ann['Male']).replace(0, np.nan)
# print(pat_per_ann[['Female','Male','female_ratio']].head())

# Resultado final:
assignments = assignments_df

# Se quiser inspecionar:
# print(assignments.head())
# print(assignments.shape)

In [4]:
assignments.to_csv(OUTPUT_PATH, index=False)

In [5]:
assignments

,annotator_id,frase_id
0,1,1905201
1,1,4278896
2,1,3090531
3,1,2056355
4,1,1971942
...,...,...
69995,35,2304718
69996,35,1864909
69997,35,4211722
69998,35,994935
